In [ ]:
from pathlib import Path

from astropy.io import fits
from astropy.io.fits.fitsrec import FITS_rec
from numpy.typing import NDArray
import numpy as np
from scipy.interpolate import griddata

In [ ]:
def load_fits_data(path: Path, ext: int = 1) -> FITS_rec:
    """Loads the FITS file data from chosen extension."""
    return fits.getdata(path, ext=ext, header=False)

def extract_spectrum(
    data: FITS_rec,
    source_idx: int,
    energy_range: tuple[float | None, float | None] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts source energy and spectrum values in specified `energy_range`.
    """
    energy: NDArray = data['ENERGY'][source_idx]
    energy_ = energy[:-1] + np.diff(energy) / 2

    low, high = energy_range
    band: NDArray = (energy_ > low) & (energy_ < high)
    spectrum: NDArray = data['SPECTRUM'][source_idx]

    return energy_[band], spectrum[band]

def interp_from_energy(
    energy: NDArray,
    spectrum: NDArray,
    energy_band: NDArray,
) -> NDArray:
    """Interpolates spectrum values in given `energy_band` values."""
    # ...
    # probably some preprocess
    # ...
    return griddata(energy, spectrum, energy_band)

def compute_theta(theta_x: float, theta_y: float) -> float:
    """
    Computes the polar angular coord wrt to the xy plane.
    Both `theta_x` and `theta_y` are in [deg].
    Output angle value is in [deg].
    """
    _ts = np.array([theta_x, theta_y])
    _ts = np.square(np.tan(np.deg2rad(_ts)))
    _ts = np.sqrt(_ts.sum())
    return np.rad2deg(np.atan(_ts))

In [ ]:
settings_path: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/camera_settings"

sources_path: Path = Path(settings_path, "RXTE-ASM_BeppoSAX-WFC_catalog_new_2-50keV.fits")

detSi_matten_path: Path = Path(settings_path, "detectorSi_absrp.fits")
detBe_dl_filter_path: Path = Path(settings_path, "detectorBe_deadlayer_filter.fits")

maskKapton_mli_path: Path = Path(settings_path, "mask_MLI_Kapton.fits")

sources: FITS_rec = load_fits_data(sources_path)
detSi_matten: FITS_rec = load_fits_data(detSi_matten_path)
detBe_dl_filter: FITS_rec = load_fits_data(detBe_dl_filter_path)
maskKapton_mli: FITS_rec = load_fits_data(maskKapton_mli_path)

**Source Energy Band Counts**

In [ ]:
energy, spectrum = extract_spectrum(sources, source_idx=0)

In [89]:
def extract_transmission(
    data: FITS_rec,
    energy_range: tuple[float | None, float | None] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts photons transmission values in specified `energy_range`.
    """
    energy: NDArray = data.field(0)
    transmission: NDArray = data.field(1)
    low, high = energy_range
    band: NDArray = (energy > low) & (energy < high)
    return energy[band], transmission[band]


e, t = extract_transmission(detSi_matten)
e[0], e[-1], t[0 : 5], t[-1]

(np.float32(2.006396),
 np.float32(49.80478),
 array([2752.275 , 2724.685 , 2697.3718, 2670.3323, 2643.564 ], dtype='>f4'),
 np.float32(0.2335848))

In [ ]:
import darksun as ds

def func(x: NDArray) -> NDArray:
    return np.pow(x, 2) * np.exp(-x / 5)


a = np.linspace(0, 25, 26)
f = func(a)

xi = np.sort(np.random.uniform(0, 25, 10))

griddata(a, f, xi), func(xi)

x = np.linspace(0, 25, 501)
ds.plot(
    ds.map4plot(
        arrs=(func(x), func(xi), griddata(a, f, xi)),
        title='',
        labels=('func', 'true', 'interp'),
        x=(x, xi, xi),
        style='scatter',
    ),
)